In [ ]:
# CNN stands for Convolutional Neural Networks, it basically consists of convolutional layers, pooling layers, and fully connected layers. It is a class of deep neural networks, most commonly applied to analyzing visual imagery.
# in simple words, CNN is a type of deep learning model that is designed to process and analyze visual data, such as images and videos. It is inspired by the structure and function of the human visual system, which is able to recognize and interpret complex visual patterns.

import torch
import torch.nn as nn

image = torch.randn(1, 1, 28, 28)  # Example input image with shape (batch_size, channels, height, width)
conv=nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)  # input channel means the number of channels in the input image, output channel means the number of filters applied to the input image

output = conv(image)
print("Input shape:", image.shape)  # Input shape will be (batch_size, in_channels, height, width)
print("Output shape:", output.shape)  # Output shape will be (batch_size, out_channels, height, width)


Input shape: torch.Size([1, 1, 28, 28])
Output shape: torch.Size([1, 8, 28, 28])


In [ ]:
# Pooling selects the most important features from the feature map, 
# allowing the network to focus on the most relevant information while discarding less important details.

import torch
import torch.nn as nn

x = torch.randn(1, 8, 28, 28)  # Example input image with shape (batch_size, channels, height, width)
pool=nn.MaxPool2d(kernel_size=2)  # Max pooling operation with a 2x2 kernel 

y=pool(x)
print("Before Pooling:", x.shape)  # Input shape will be (batch_size, in_channels, height, width)
print("After Pooling:", y.shape)  # Output shape will be (batch_size, out_channels, height, width)


Before Pooling: torch.Size([1, 8, 28, 28])
After Pooling: torch.Size([1, 8, 14, 14])


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform=transforms.ToTensor()

train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(in_features=16*7*7, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predicted = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item() 

print(f"Accuracy: {100 * correct / total:.2f}%")
torch.save(model.state_dict(), "simple_cnn.pth")

Using device: cpu
Epoch [1/3], Loss: 0.0761
Epoch [2/3], Loss: 0.0228
Epoch [3/3], Loss: 0.0142
Accuracy: 98.30%
